In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

embedding_model = DefaultEmbeddingFunction()  # local all-MiniLM-L6-v2, no API key

def get_embeddings(texts):
    """Convert text to embedding vectors."""
    if isinstance(texts, str):
        texts = [texts] 

    vectors = embedding_model(texts)
    return np.array(vectors)


sentences = [
    "The cat is sleeping on the couch",
    "A kitten is playing with a toy",
    "The dog is running in the park"
]
embeddings = get_embeddings(sentences)

cat_kitten = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]  #A
cat_dog = cosine_similarity([embeddings[0]], [embeddings[2]])[0][0]  #A

print(f"Cat vs Kitten: {cat_kitten:.3f}")
print(f"Cat vs Dog: {cat_dog:.3f}")


Cat vs Kitten: 0.374
Cat vs Dog: 0.088


texts with similar meaning land close together and point in similar directions;
unrelated texts point in different directions

In [ ]:
def fixed_length_chunking(text, chunk_size=500, overlap=50):
    """Split text into fixed-length chunks."""
    if chunk_size <= 0 or not 0 <= overlap < chunk_size:
        raise ValueError("Require chunk_size > 0 and 0 <= overlap < chunk_size")
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap if end < len(text) else end  #A

    return chunks


sample = "A" * 283
chunks = fixed_length_chunking(sample, chunk_size=100, overlap=20)
print(f"Original: {len(sample)} chars → {len(chunks)} chunks")



Original: 283 chars → 4 chunks


vector search


In [7]:
def vector_search(query, chunks, chunk_embeddings, top_k=3):
    """Find the most similar chunks to the query."""
    query_embedding = get_embeddings(query)
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]
    top_indices = similarities.argsort()[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            'chunk': chunks[idx],
            'similarity': similarities[idx]
        })
    return results


documents = [
    "Python is a programming language",
    "Machine learning uses Python extensively",
    "Cats are popular pets",
    "Deep learning is a subset of machine learning"
]

doc_embeddings = get_embeddings(documents)

results = vector_search("Artificial Intelligence", documents, doc_embeddings, top_k=4)
for r in results:
    print(f"{r['similarity']:.3f}: {r['chunk']}")



0.415: Deep learning is a subset of machine learning
0.385: Machine learning uses Python extensively
0.311: Python is a programming language
0.045: Cats are popular pets


end-to-end vector search
Full pipeline that fetches web search results, chunks and embeds them, uses vector search to extract only the most relevant content

In [8]:
from tavily import TavilyClient

tavily = TavilyClient()
response = tavily.search(
    "2025 Nobel Prize winners",
    max_results=10,
    include_raw_content=True
)

search_results = []
for result in response['results']:
    if result.get('raw_content'):
        search_results.append({
            'title': result['title'],
            'content': result['raw_content'],
            'url': result['url']
        })


In [9]:
import tiktoken

try:
    enc = tiktoken.encoding_for_model("gpt-5.5")
except KeyError:
    enc = tiktoken.get_encoding("o200k_base")
full_text = "\n\n".join([
    f"Title: {r['title']}\n{r['content']}"
    for r in search_results
])
total_tokens = len(enc.encode(full_text))
print(f"Total characters: {len(full_text)}")
print(f"Total tokens: {total_tokens}")


Total characters: 181095
Total tokens: 78369


In [11]:
all_chunks = []
for result in search_results:
    text = f"Title: {result['title']}\n{result['content']}"
    chunks = fixed_length_chunking(text, chunk_size=500, overlap=50)
    for chunk in chunks:
        all_chunks.append({
            'text': chunk,
            'title': result['title'],
            'url': result['url']
        })

print(f"Total chunks: {len(all_chunks)}")

chunk_texts = [c['text'] for c in all_chunks]
chunk_embeddings = get_embeddings(chunk_texts)

query = "quantum computing"
results = vector_search(query, chunk_texts, chunk_embeddings, top_k=3)

print(f"Query: '{query}'\n")
print("=" * 60)
for i, r in enumerate(results, 1):
    print(f"\n[{i}] Similarity: {r['similarity']:.3f}")
    print(f"{r['chunk'][:300]}...")



Total chunks: 406
Query: 'quantum computing'


[1] Similarity: 0.720
the way that century-old quantum mechanics continually offers new surprises,” says Olle Eriksson, Chair of the Nobel Committee for Physics. “It is also enormously useful, as quantum mechanics is the foundation of all digital technology.”

Their work has laid the groundwork to construct quantum compu...

[2] Similarity: 0.541
nternational Year of Quantum Science and Technology](https://www.weforum.org/stories/2025/09/cybersecurity-in-the-quantum-age-and-other-frontier-technologies/), one of the prizes went to three pioneering physicists for [their work in quantum mechanics](https://www.nobelprize.org/prizes/physics/2025/...

[3] Similarity: 0.538
ng the Quantum Economy report. And that’s in part thanks to three scientists who made a discovery back in the 1980s.

#### Have you read?

This year’s Physics laureates, John Clarke, Michel H. Devoret and John M. Martinis, conducted experiments with an electrical circuit in wh

In [12]:
top_chunks = [r['chunk'] for r in results]
selected_text = "\n\n".join(top_chunks)
selected_tokens = len(enc.encode(selected_text))

print(f"Total tokens: {total_tokens}")
print(f"Selected tokens: {selected_tokens}")
print(f"Savings rate: {(1 - selected_tokens/total_tokens)*100:.1f}%")


Total tokens: 78369
Selected tokens: 303
Savings rate: 99.6%


Knowledge-based search tool
Create tools thath enable an agent to expolore and analyze files and folders - extract archives, listing files, reading various files incl text, spreadsheets, PDFs. Crawl files systhematically.
    1. Extract files
    2. Look around folders and inspect what matters

In [14]:
from datasets import load_dataset
from huggingface_hub import snapshot_download
from pathlib import Path

# Load dataset
dataset = load_dataset("gaia-benchmark/GAIA", "2023_all", split="validation")

# Download attached files
NOTEBOOK_DIR = Path.cwd()                 # D:\Code\AI-Agents\notebooks\agents_basics
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]    # D:\Code\AI-Agents
CACHE_DIR = PROJECT_ROOT / "data" / "gaia"

snapshot_download(
    repo_id="gaia-benchmark/GAIA",
    repo_type="dataset",
    allow_patterns="2023/validation/*",
    local_dir=CACHE_DIR,
)


Fetching 42 files: 100%|██████████| 42/42 [00:04<00:00,  9.14it/s]


'D:\\Code\\AI-Agents\\data\\gaia'

In [15]:
import shutil

#Resetting workspace
WORK_DIR = NOTEBOOK_DIR / "gaia_workspace"

def reset_workspace():
    """Restore the workspace to its initial state."""
    shutil.rmtree(WORK_DIR, ignore_errors=True)
    shutil.copytree(CACHE_DIR / "2023/validation", WORK_DIR)
    print(f"Workspace reset: {WORK_DIR}")

reset_workspace()


Workspace reset: d:\Code\AI-Agents\notebooks\agents_basics\gaia_workspace


In [17]:
#Finding problems with .zip files
problems_with_files = [p for p in dataset if p.get('file_name')]
problem_with_zip = [p for p in problems_with_files if p['file_name'].endswith('.zip')]

print(f"Total problems: {len(dataset)}")
print(f"Problems with attachments: {len(problems_with_files)}")
print(f"Total problems with zip files: {len(problem_with_zip)}")

if not problem_with_zip:
    raise ValueError("The selected dataset has no ZIP attachments")
problem = problem_with_zip[0]
print(f"Question: {problem['Question'][:100]}...")
print(f"File name: {problem['file_name']}")

file_path = WORK_DIR / problem['file_name']
print(f"File exists: {file_path.exists()}")


Total problems: 165
Problems with attachments: 38
Total problems with zip files: 2
Question: Which of the text elements under CATEGORIES in the XML would contain the one food in the spreadsheet...
File name: 9b54f9d9-35ee-4a14-b62f-d130ea00317f.zip
File exists: True


In [19]:
#unzip extract archives

import zipfile
from pathlib import Path

def unzip_file(zip_path: str, extract_to: str = None) -> str:
    """Extract a zip file to the specified directory."""
    zip_path = Path(zip_path)

    if not zip_path.exists():
        return f"File not found: {zip_path}"

    # Default extraction path: create folder with zip filename
    if extract_to is None:
        extract_to = zip_path.parent / zip_path.stem
    else:
        extract_to = Path(extract_to)

    extract_to.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        file_list = zip_ref.namelist()
        zip_ref.extractall(extract_to)

    # Format results
    result = f"Extracted {len(file_list)} files to {extract_to}/\n\n"
    result += "Contents:\n"
    for f in file_list[:20]:
        result += f"  - {f}\n"
    if len(file_list) > 20:
        result += f"  ... and {len(file_list) - 20} more files\n"

    return result


In [20]:
#implement list files

from pathlib import Path

def list_files(path: str = ".") -> str:
    """List files and directories in the given path."""
    path = Path(path)

    if not path.exists():
        return f"Path not found: {path}"

    if not path.is_dir():
        return f"Not a directory: {path}"

    items = []
    for item in sorted(path.iterdir()):
        if item.name.startswith('.'):
            continue

        if item.is_dir():
            items.append(f"{item.name}/")
        else:
            items.append(f"{item.name}")

    # Sort directories first
    dirs = [i for i in items if i.endswith('/')]
    files = [i for i in items if not i.endswith('/')]

    result = f"Directory: {path}\n"
    for item in dirs + files:
        result += f"  {item}\n"

    return result


In [21]:
#implement read file allowinf the agent to read text, CSV, JSON

from pathlib import Path

TEXT_EXTENSIONS = ['.txt', '.py', '.js', '.json', '.md', '.html',
                   '.css', '.xml', '.yaml', '.yml', '.log', '.sh']
SPREADSHEET_EXTENSIONS = ['.xlsx', '.xls', '.csv']

def read_file(file_path: str, start_line: int = 1, end_line: int = -1) -> str:
    """Read file content. Supports txt, py, json, md, csv, xlsx."""
    path = Path(file_path)

    if not path.exists():
        return f"File not found: {file_path}"

    ext = path.suffix.lower()

    if ext in TEXT_EXTENSIONS:
        return _read_text_file(file_path, start_line, end_line)
    elif ext == '.csv':
        return _read_csv(file_path)
    elif ext in SPREADSHEET_EXTENSIONS:
        return _read_excel(file_path)
    else:
        return _read_text_file(file_path, start_line, end_line)


In [22]:
#reading text file with line numbers

def _read_text_file(file_path: str, start_line: int, end_line: int) -> str:
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # Adjust line numbers (1-indexed to 0-indexed)
    start_idx = max(0, start_line - 1)
    end_idx = len(lines) if end_line == -1 else min(end_line, len(lines))

    selected_lines = lines[start_idx:end_idx]

    result = []
    for i, line in enumerate(selected_lines, start=start_line):
        result.append(f"{i:4d} | {line.rstrip()}")
    return '\n'.join(result)


In [23]:
# convertint to .md tables to aid LLM understanding
import pandas as pd

def _read_csv(file_path: str) -> str:
    df = pd.read_csv(file_path)
    return df.to_markdown(index=False)

def _read_excel(file_path: str) -> str:
    df = pd.read_excel(file_path)
    return df.to_markdown(index=False)



In [24]:
#read media files

import fitz  # pymupdf
import base64
from pathlib import Path
from openai import OpenAI

IMAGE_EXTENSIONS = ['.png', '.jpg', '.jpeg', '.gif', '.webp', '.bmp']
AUDIO_EXTENSIONS = ['.mp3', '.wav', '.m4a', '.flac', '.ogg', '.webm']
PDF_EXTENSIONS = ['.pdf']

def read_media_file(file_path: str, query: str) -> str:
    """Analyze an image, audio, or PDF file using LLM."""
    ext = Path(file_path).suffix.lower()

    if ext in IMAGE_EXTENSIONS:
        return _analyze_image(file_path, query)
    elif ext in AUDIO_EXTENSIONS:
        return _analyze_audio(file_path, query)
    elif ext in PDF_EXTENSIONS:
        return _analyze_pdf(file_path, query)
    else:
        return f"Unsupported media format: {ext}"

def _analyze_image(file_path: str, query: str) -> str:
    with open(file_path, "rb") as f:
        image_data = base64.b64encode(f.read()).decode("utf-8")

    ext = Path(file_path).suffix.lower().lstrip('.')
    media_type = "image/jpeg" if ext == "jpg" else f"image/{ext}"

    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-5.5",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": query},
                {"type": "image_url", "image_url": {
                    "url": f"data:{media_type};base64,{image_data}"
                }}
            ]
        }]
    )
    return response.choices[0].message.content

def _analyze_audio(file_path: str, query: str) -> str:
    with open(file_path, "rb") as f:
        audio_data = base64.b64encode(f.read()).decode("utf-8")

    ext = Path(file_path).suffix.lower().lstrip('.')

    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-audio",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": query},
                {"type": "input_audio", "input_audio": {
                    "data": audio_data,
                    "format": ext
                }}
            ]
        }]
    )
    return response.choices[0].message.content

def _analyze_pdf(file_path: str, query: str) -> str:
    doc = fitz.open(file_path)

    # Extract text for context
    text_content = ""
    for page in doc:
        text_content += page.get_text()

    # Convert pages to images
    images = []
    for page in doc[:5]:  # First 5 pages
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        img_bytes = pix.tobytes("png")
        images.append(base64.b64encode(img_bytes).decode('utf-8'))

    # Build content with text and images
    content = [{
        "type": "text",
        "text": f"{query}\n\nExtracted text:\n{text_content[:3000]}"
    }]

    for img_b64 in images:
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{img_b64}"}
        })

    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-5.5",
        messages=[{"role": "user", "content": content}]
    )
    return response.choices[0].message.content


In [ ]:
#Register tools with the Agent

from scratch_agents.tools.base import tool
from scratch_agents.agent import Agent
from scratch_agents.llm import LlmClient
from scratch_agents.tools.search import search_web

# File system tools — wrap the functions defined in Listings 5.15–5.20
unzip_file_tool = tool(unzip_file)
list_files_tool = tool(list_files)
read_file_tool = tool(read_file)
read_media_file_tool = tool(read_media_file)

# Configure with existing tools
tools = [
    search_web,
    unzip_file_tool,
    list_files_tool,
    read_file_tool,
    read_media_file_tool,
]

agent = Agent(
    model=LlmClient(model="gpt-5.5"),
    tools=tools,
    instructions="You are a helpful assistant that can search the web and "
                "explore files to answer questions.",
    max_steps=20,
)


Solving ZIP GAIA problems

In [25]:
# Reset workspace to clean state
reset_workspace()

# Select a problem with zip file attachment
zip_problems = [p for p in problems_with_files if p['file_name'].endswith('.zip')]
if len(zip_problems) < 2:
    raise ValueError("This listing requires the second ZIP attachment from the GAIA dataset")
problem = zip_problems[1]

file_path = WORK_DIR / problem['file_name']

# Construct prompt including file location
prompt = f"""{problem['Question']}

The attached file is located at: {file_path}
"""
print(prompt)


Workspace reset: d:\Code\AI-Agents\notebooks\agents_basics\gaia_workspace
How many applicants for the job in the PDF are only missing a single qualification?

The attached file is located at: d:\Code\AI-Agents\notebooks\agents_basics\gaia_workspace\bfcd99e1-0690-4b53-a85c-0174a8629083.zip



Adding new behaviours - like pausing for human approval before deleting files

In [ ]:
**Implementation excerpt** — read with the surrounding explanation; the complete implementation is in `scratch_agents/`.

```python
class Agent:
    def __init__(
        self,
        model: LlmClient,
        tools: List[BaseTool] = None,
        instructions: str = "",
        max_steps: int = 10,
        output_type: Optional[Type[BaseModel]] = None,
        before_tool_callbacks: List[Callable] = None,
        after_tool_callbacks: List[Callable] = None,
    ):
        self.model = model
        self.instructions = instructions
        self.max_steps = max_steps
        self.output_type = output_type
        self.output_tool_name: str | None = None
        self.tools = self._setup_tools(tools or [])

        # Initialize callback lists
        self.before_tool_callbacks = before_tool_callbacks or []
        self.after_tool_callbacks = after_tool_callbacks or []
```
